# Interactive Notebook 03 - PMSM parameter identification:

This interactive Jupyter notebook introduces common parameter identification principles for permanent magnet synchronous motors.

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})

import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

In [ ]:
from helper_functions import visualize_eigendynamics, estimate_eigendynamics_grid, visualize_trajectories, aprbs

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.)

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---

**Throught the whole notebook, we will assume that the rotation speed $\omega$ can be set arbitrarily from an external load machine.**

### Simulation of the standard linear PMSM with constant inductances:

The standard ODE for the linear PMSM with constant inductances in dq-coordinates (see lecture 3 eq. 3.10) is

$$ \large \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{i}_{\mathrm{s, dq}} (t) = \begin{bmatrix} L^{-1}_{\mathrm{dd}, 0} & 0 \\ 0 & L^{-1}_{\mathrm{qq}, 0}  \end{bmatrix} 
\left[ \boldsymbol{u}_{\mathrm{s, dq}} (t) - R_\mathrm{s} \boldsymbol{i}_{\mathrm{s, dq}} (t) - \omega (t) \boldsymbol{J} \left( \begin{bmatrix} 0 \\ \psi_\mathrm{PM} \end{bmatrix} + \begin{bmatrix} L_{\mathrm{dd}, 0} & 0 \\ 0 & L_{\mathrm{qq}, 0}  \end{bmatrix}  \boldsymbol{i}_{\mathrm{s, dq}} (t) \right)  \right] 
$$

To simulate the electrical motor behavior, we first define the necessary parameters as

$$ \large  \boldsymbol{\theta} = \begin{bmatrix} p & R_\mathrm{s} & L_\mathrm{dd} & L_\mathrm{qq} & \psi_\mathrm{PM} \end{bmatrix}.$$


In [ ]:
class LinearParams(eqx.Module):
    p: jax.Array = eqx.field(converter=jnp.asarray)  # Number of pole pairs
    R_s: jax.Array = eqx.field(converter=jnp.asarray)  # Stator resistance
    L_dd: jax.Array = eqx.field(converter=jnp.asarray)  # D-axis inductance
    L_qq: jax.Array = eqx.field(converter=jnp.asarray)  # Q-axis inductance
    psi_pm: jax.Array = eqx.field(converter=jnp.asarray)  # Permanent magnet flux linkage

linear_motor_params = LinearParams(
    p=3,
    R_s=jnp.array(17.932e-3),
    L_dd=jnp.array(0.37e-3),
    L_qq=jnp.array(1.2e-3),
    psi_pm=jnp.array(65.65e-3), 
)

The ODE itself can be implemented as:

In [ ]:
@eqx.filter_jit
def linear_ode(i_dq_t, u_dq_t, omega_t, params):

    L_inv = jnp.array([[1/params.L_dd, 0],[0, 1/params.L_qq]])
    J = jnp.array([[0, -1],[1, 0]])
    psi_pm_vec = jnp.array([ params.psi_pm, 0])
    L = jnp.array([[params.L_dd, 0],[0, params.L_qq]])
    
    return L_inv @ (u_dq_t - params.R_s * i_dq_t - omega_t * J @ (psi_pm_vec + L @ i_dq_t))

#### Visualization of the Eigendynamics as a vectorfield:

Without any mechanical rotation, the currents 

In [ ]:
fig, axs = visualize_eigendynamics(ode=linear_ode, params=linear_motor_params, n_values=[0])

With higher rotation speed the EMF and the Eigendynamics increase in strength:

In [ ]:
fig, axs = visualize_eigendynamics(ode=linear_ode, params=linear_motor_params, n_values=[0, 2500, 5000, 7500, 10000])

#### Explicit Euler:

To simulate trajectories from the model, we require an ODE solver.
As discussed in the lecture, the simplest form is to use explicit Euler

$$ \large \boldsymbol{i}_{\mathrm{s, dq}}[k+1] = \boldsymbol{i}_{\mathrm{s, dq}}[k] + T_\mathrm{s} \boldsymbol{f}_{\boldsymbol{\theta}} (\boldsymbol{i}_{\mathrm{s, dq}}[k], \boldsymbol{u}_{\mathrm{s, dq}}[k], \omega[k]), $$
where 
$$ \large \boldsymbol{f}_{\boldsymbol{\theta}} (\boldsymbol{i}_{\mathrm{s, dq}}[k], \boldsymbol{u}_{\mathrm{s, dq}}[k], \omega[k]) =  \begin{bmatrix} L^{-1}_{\mathrm{dd}, 0} & 0 \\ 0 & L^{-1}_{\mathrm{qq}, 0}  \end{bmatrix} 
\left[ \boldsymbol{u}_{\mathrm{s, dq}}[k] - R_\mathrm{s} \boldsymbol{i}_{\mathrm{s, dq}}[k]- \omega[k] \boldsymbol{J} \left( \begin{bmatrix} 0 \\ \psi_\mathrm{PM} \end{bmatrix} + \begin{bmatrix} L_{\mathrm{dd}, 0} & 0 \\ 0 & L_{\mathrm{qq}, 0}  \end{bmatrix}  \boldsymbol{i}_{\mathrm{s, dq}}[k] \right)  \right] . $$

In [ ]:
def linear_euler_step(i_dq, u_dq, omega, params, T_s):
    return i_dq + T_s * linear_ode(i_dq, u_dq, omega, params)

@eqx.filter_jit
def linear_euler_simulation(i_dq_0, u_dq_sequence, omega, params, T_s):

    def body_function(carry, u_dq):
        i_dq = carry
        i_dq_next = linear_euler_step(i_dq, u_dq, omega, params, T_s)
        return i_dq_next, i_dq_next

    _, i_dq_sequence =  jax.lax.scan(body_function, i_dq_0, u_dq_sequence)
    i_dq_sequence = jnp.concatenate([i_dq_0[None, :], i_dq_sequence], axis=0)
    return i_dq_sequence

In [ ]:
sequence_length = 10_000
T_s = 1e-4
n = 1000
u_dq_sequence = jnp.zeros((sequence_length, 2))

omega = jnp.array(linear_motor_params.p * n * 2 * jnp.pi / 60)

i_dq_sequence = linear_euler_simulation(
    i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=linear_motor_params, T_s=T_s,
)

visualize_trajectories(
    [i_dq_sequence], [u_dq_sequence], [T_s], [omega], linear_ode, linear_motor_params,
)

In [ ]:
i_dq_sequences = []
u_dq_sequences = []
omegas = []

T_s_list = [1e-5, 1e-4, 1e-3, 1e-2]

for T_s in T_s_list:
    i_dq_sequence = linear_euler_simulation(
        i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=linear_motor_params, T_s=T_s,
    )

    i_dq_sequences.append(i_dq_sequence)
    u_dq_sequences.append(u_dq_sequence)
    omegas.append(omega)

visualize_trajectories(
    i_dq_sequences, u_dq_sequences, T_s_list, omegas, linear_ode, linear_motor_params, labels=[f"$T_\\mathrm{{s}}$={T_s}" for T_s in T_s_list]
)

With small time steps we can get the expected behavior, but one has to be careful not too choose it to large. 

**Maybe replace with an exemplary operating point, that is actually stable?**

In [ ]:
sequence_length = 10_000
T_s = 1e-4
n = 500
u_dq_sequence = aprbs(sequence_length, 2, t_min=1000, t_max=3000, key=jax.random.key(0)).T[0] * 50

omega = jnp.array(linear_motor_params.p * n * 2 * jnp.pi / 60)

i_dq_sequence = linear_euler_simulation(
    i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=linear_motor_params, T_s=T_s,
)

visualize_trajectories(
    [i_dq_sequence], [u_dq_sequence], [T_s], [omega], linear_ode, linear_motor_params,
)

#### Advanced ODE solvers:

Next we transfer the ODE into the API of the diffrax Python library. 
This way we can use the advanced ODE solvers that come with the library without having to implement them ourselves.

In [ ]:
def diffrax_linear_ode(t, y, args, action):
    i_dq = y
   
    params, omega = args
    u_dq = action(t)

    return linear_ode(i_dq, u_dq, omega, params)

In [ ]:
@eqx.filter_jit
def linear_diffrax_simulation(i_dq_0, u_dq_sequence, omega, params, T_s, solver, adaptive_stepsize=False):

    def voltage(t):
        return u_dq_sequence[jnp.array(t / T_s, int)]
    
    args = (linear_motor_params, omega)
    vector_field = partial(diffrax_linear_ode, action=voltage)
    
    term = diffrax.ODETerm(vector_field)
    t0 = 0
    t1 = T_s * u_dq_sequence.shape[0]
    
    y0 = i_dq_0
    saveat = diffrax.SaveAt(ts=jnp.linspace(t0, t1, 1 + int(t1 / T_s)))

    if adaptive_stepsize:
        stepsize_controller = diffrax.PIDController()
    else:
        stepsize_controller = diffrax.ConstantStepSize()
    
    y = diffrax.diffeqsolve(
        term,
        solver,
        t0,
        t1,
        dt0=T_s,
        y0=y0,
        args=args,
        saveat=saveat,
        stepsize_controller=stepsize_controller,
        max_steps=100_000
    )
    return y.ys

In [ ]:
T_s = 1e-3
n = 1000
sequence_length = 1_000

u_dq_sequence = jnp.zeros((sequence_length, 2))

i_dq_sequences = []
u_dq_sequences = []
omegas = []

solver_list = [diffrax.Euler(), diffrax.Heun(), diffrax.Tsit5()]

for solver in solver_list:
    i_dq_sequence = linear_diffrax_simulation(
        i_dq_0=jnp.array([0.0, 0.0]),
        u_dq_sequence=u_dq_sequence,
        omega=omega,
        params=linear_motor_params,
        T_s=T_s,
        solver=solver,
    )

    i_dq_sequences.append(i_dq_sequence)
    u_dq_sequences.append(u_dq_sequence)
    omegas.append(omega)

visualize_trajectories(
    i_dq_sequences,
    u_dq_sequences,
    [T_s for _ in range(len(solver_list))],
    omegas,
    linear_ode,
    linear_motor_params,
    labels=[r"$\mathrm{Euler}$", r"$\mathrm{Heun}$", r"$\mathrm{Tsit5}$"],
)

Heun and Tsit5 stay stable, while Euler diverges. Heun and Tsit5 require more computational effort though.
In essence, the main advantage of higher order methods comes to play when adaptive stepsizing is used:

In [ ]:
raise NotImplementedError()

### Nonlinear PMSM model:

- flux maps
- compare behavior to linear case

In [ ]:
raise NotImplementedError()